In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain import Rain

sys.path.pop()

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [2]:
import sys
sys.path.append('../../../')
from clean_all import clean
clean()
sys.path.pop()

'../../../'

In [3]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions

In [4]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        # network parameters
        hidden_units = 256
        dropout = 0.45
        input_size = 784
        num_labels = 10
        # Define the layers
        self.fc1 = nn.Linear(input_size, hidden_units)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_units, hidden_units)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden_units, num_labels)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

In [5]:
model = Model()
config = {
    "lib": "pytorch",
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 20,
    "batch_size": 128,
    "loss": nn.CrossEntropyLoss(),
    "optimizer": optim.Adam(model.parameters(), lr=0.001)
}

In [6]:
X_train, y_train = get_train_data()
y_train = np.argmax(y_train, axis=1)

# Partition training data
X_train, y_train = partition_train_data(X_train, y_train, config['partitions'])

In [7]:
for i in range(len(X_train)):
    np.save(f"../../../data/X_train_{i + 1}.npy", X_train[i])
    np.save(f"../../../data/y_train_{i + 1}.npy", y_train[i])

In [8]:
rain = Rain(config, model, X_train, y_train)

Rain is initialized
Provisioner created successfully
divider is running


In [9]:
model = rain.train_centralized_sync()

divider received: Success receiving the number of workers
divider is sending data to the coordinator
 divider received: File received successfully
 divider received: File received successfully
 divider received: File received successfully
 divider received: File received successfully
 divider received: File received successfully
 divider received: File received successfully
Starting iteration 1/3
sending file:  ../../../Divider/divider/data/1.pkl
divider is sending information file to the coordinator
 divider received: File received successfully
divider begins the iteration
 divider received: one loop is done
Iteration 1/3 complete.
Starting iteration 2/3
sending file:  ../../../Divider/divider/data/2.pkl
divider is sending information file to the coordinator
 divider received: File received successfully
divider begins the iteration
filepath is: ../../../Divider/divider/data/2_2_trained.pkl
filepath is: ../../../Divider/divider/data/3_2_trained.pkl
 divider received: one loop is done
I

In [10]:
def evaluate_model(model, X_test, y_test, batch_size):
    # Convert numpy arrays to PyTorch tensors
    X_test = torch.from_numpy(X_test).float()
    y_test = torch.from_numpy(y_test).long()

    # Create a TensorDataset
    test_dataset = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    total_correct = 0
    total_samples = 0
    
    for i, (data, labels) in enumerate(test_loader):
        # Forward pass
        outputs = model(data)

        # Compute training accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_correct += (predicted == labels).sum().item()
        total_samples += labels.size(0)

    return total_correct / total_samples

In [11]:
X_test, y_test = get_test_data()
y_test = np.argmax(y_test, axis=1)
acc = evaluate_model(model, X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))


Test accuracy: 97.4%
